# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their @ids
record_sets = []
print("Available record sets and their '@id':\n")
for rset in dataset.metadata.record_sets:
    record_sets.append(rset["@id"])
    print(f"- Record Set: {rset['@id']} | name: {rset.get('name','')}")
    if 'fields' in rset:
        print("  Fields:")
        for field in rset['fields']:
            print(f"    - {field['@id']} | {field.get('name','')}")
print("\nExample records from the first record set:")
if record_sets:
    example_records = dataset.records(record_set=record_sets[0])
    for i, rec in enumerate(example_records):
        if i >= 2:
            break
        print(json.dumps(rec, indent=2))
else:
    print('No record sets found.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
dataframes = {}

# Display available record sets
print("Loaded record sets (by @id):", record_sets)

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"\nRecord set '{record_set}' fields:", df.columns.tolist())

# Select the main record set for analysis. If only one, use it. Customize as needed.
main_record_set = record_sets[0] if record_sets else None

if main_record_set and not dataframes[main_record_set].empty:
    display(dataframes[main_record_set].head())
else:
    print('No data found in the main record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, we will try to select a numeric field.
# You should replace these with known @id from field list or column names seen above.
import numpy as np
if main_record_set and not dataframes[main_record_set].empty:
    df = dataframes[main_record_set]
    # Heuristically pick a numeric field with float or int type
    numeric_candidate = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break
    if numeric_candidate is None:
        # Try to coerce a likely-numeric column
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_candidate = col
                    break
            except:
                continue

    print(f"Using numeric field: {numeric_candidate}")
    if numeric_candidate:
        threshold = np.nanmean(df[numeric_candidate])
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_candidate}_normalized"] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
        print(f"Normalized {numeric_candidate} for filtered records:")
        display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Try grouping by another field (string/object type)
        group_field = None
        for col in df.columns:
            if col != numeric_candidate and df[col].dtype == 'object':
                group_field = col
                break
        print(f"Candidate group field: {group_field}")
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
    else:
        print('No numeric field found for EDA.')
else:
    print('No available main DataFrame for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and not dataframes[main_record_set].empty and numeric_candidate:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_candidate].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_candidate}')
    plt.xlabel(numeric_candidate)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_candidate, data=df)
        plt.title(f'{numeric_candidate} grouped by {group_field}')
        plt.ylabel(numeric_candidate)
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Not enough numeric/grouped data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and inspected the Croissant dataset using `mlcroissant`.
- Record sets, fields, and IDs were reviewed; a main record set was selected for analysis.
- Numeric fields were filtered, normalized, and visualized; grouping provided insight into categorical structure.
- This approach can be extended or customized for specific research questions related to the dataset, such as analyzing protein abundance or modifications.

**Next Steps:**
- Integrate domain-specific filters or metrics.
- Combine with other experimental metadata for multi-modal analysis.
- Build downstream pipelines for machine learning or statistical modeling.